# What This Walkthrough Does

This capstone-style notebook puts the pieces together:

- explicit configuration
- folder structure
- API request
- raw output
- processed table
- manifest
- limitations

## Packages Used in This Notebook

This walkthrough uses:

- `requests` to collect data from the MediaWiki API
- `pandas` to create processed tables and save small JSON files
- `Path` to keep raw, processed, and report outputs separate
- `datetime` and `timezone` to timestamp the manifest

The point is not only to collect data, but to package the collection as a reproducible mini-project.


In [ ]:
# datetime/timezone let us create an unambiguous UTC timestamp for the manifest.
from datetime import datetime, timezone
# Path helps create output folders and file paths.
from pathlib import Path

# pandas turns extracted rows into tables and helps save small JSON records.
import pandas as pd
# requests sends the API request.
import requests


# Configuration

Configuration makes parameters visible and reviewable.

In [ ]:
config = {
    "project_name": "notebook_reproducible_api_project",
    "endpoint": "https://en.wikipedia.org/w/api.php",
    "query": "platform governance",
    "pages": 1,
    "page_size": 10,
    "outdir": "data",
}

print(config)


# Prepare Folders

In [ ]:
outdir = Path(config["outdir"])

raw_dir = outdir / "raw"
processed_dir = outdir / "processed"
reports_dir = outdir / "reports"

for folder in [raw_dir, processed_dir, reports_dir]:
    # mkdir(..., parents=True, exist_ok=True) # creates the folder and any missing parent folders, without failing if it already exists.
    folder.mkdir(parents=True, exist_ok=True)


The folders encode workflow stages:

- `raw`: source-shaped evidence
- `processed`: analysis-shaped tables
- `reports`: manifests, audits, notes, provenance

# Collect One API Page

In [ ]:
params = {
    "action": "query",
    "list": "search",
    "srsearch": config["query"],
    "srlimit": config["page_size"],
    "format": "json",
    "formatversion": 2,
}

headers = {"User-Agent": "methodsNET-VLOP-course/1.0 reproducible project notebook"}

# requests.get() sends an HTTP GET request. For APIs, params become query-string fields; for web pages, it fetches HTML.
response = requests.get(config["endpoint"], params=params, headers=headers, timeout=30)
# raise_for_status() turns HTTP error responses (such as 403, 404, or 429) into visible Python errors.
response.raise_for_status()
# .json() parses a JSON response into Python dictionaries and lists. Use it only when the response actually contains JSON.
payload = response.json()

print("Request URL:", response.url)
print("Status:", response.status_code)


# Save Raw Response

In [ ]:
raw_path = raw_dir / "notebook_project_raw_response.json"

pd.Series(
    {
        "request_url": response.url,
        "status_code": response.status_code,
        "payload": payload,
    }
# to_json(...) saves structured data as JSON, which is useful for raw API responses and provenance records.
).to_json(raw_path, indent=2)

print(raw_path)


# Process Into a Table

In [ ]:
rows = []

# .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
for item in payload.get("query", {}).get("search", []):
    rows.append(
        {
            "query": config["query"],
            "pageid": item.get("pageid"),
            "title": item.get("title"),
            "timestamp": item.get("timestamp"),
            "wordcount": item.get("wordcount"),
        }
    )

# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
df = pd.DataFrame(rows)

processed_path = processed_dir / "notebook_project_processed.csv"
# to_csv(..., index=False) saves a DataFrame as a CSV file without adding pandas row numbers as a fake column.
df.to_csv(processed_path, index=False)

df.head()


# Write a Manifest

A manifest explains what was created and how (i.e., provenance)

In [ ]:
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_name": config["project_name"],
    "access_route": "MediaWiki API",
    "parameters": config,
    "raw_output": str(raw_path),
    "processed_output": str(processed_path),
    "row_count": len(df),
    "limitations": [
        "Search results are API-ranked and query-dependent.",
        "This is not a measure of all platform-governance content on Wikipedia.",
        "Only one small page of results was collected for teaching purposes.",
    ],
}

manifest_path = reports_dir / "notebook_project_manifest.json"
# to_json(...) saves structured data as JSON, which is useful for raw API responses and provenance records.
pd.Series(manifest).to_json(manifest_path, indent=2)

print(manifest_path)


# Practical Collection Workflows

The previous sections show a reproducible mini-project: configuration, raw data, processed data, and a manifest.

A practical collector that runs more than once needs a bit more infrastructure:

- **scheduling**: when and how the collector should run;
- **logging**: what happened during a run;
- **monitoring**: whether the run produced plausible output;
- **versioning**: which code, Python version, and package versions produced the output;
- **run folders**: one folder per execution, so outputs from different runs do not overwrite each other.

This is the difference between "a script that works once" and "a collection workflow someone else can audit."

## Run Folders

For repeated collection, avoid writing every run into the same files. A safer pattern is one folder per run:

```text
data/runs/
  20260703T090000Z_methodsnet_course_monitor/
    config/
    raw/
    processed/
    logs/
    reports/
```

This makes it easier to compare runs, detect breakage, and reproduce what happened.

In [ ]:
# datetime/timezone create an unambiguous UTC run timestamp.
# UTC avoids confusion when work crosses time zones or daylight-saving changes.
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_notebook_demo"

run_dir = Path(config["outdir"]) / "runs" / run_id
run_raw_dir = run_dir / "raw"
run_processed_dir = run_dir / "processed"
run_logs_dir = run_dir / "logs"
run_reports_dir = run_dir / "reports"
run_config_dir = run_dir / "config"

for folder in [run_raw_dir, run_processed_dir, run_logs_dir, run_reports_dir, run_config_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print(run_dir)

## Logging

`print()` is fine for live teaching, but scheduled collectors need logs because nobody is watching the terminal.

Python's built-in `logging` module can write timestamped messages to a file. A log should record at least:

- when the run started and ended;
- which collector/config was used;
- how many records were collected;
- warnings about empty results, failed requests, missing fields, or selector changes;
- errors that need inspection.

In [ ]:
import logging

log_path = run_logs_dir / "run.log"

# getLogger(...) creates or retrieves a named logger. A named logger is easier
# to control than using the root logger in larger projects.
logger = logging.getLogger("notebook_practical_workflow")
logger.setLevel(logging.INFO)

# Clear old handlers so re-running this notebook cell does not duplicate messages.
logger.handlers.clear()

# FileHandler writes log messages to a file.
file_handler = logging.FileHandler(log_path, encoding="utf-8")
file_handler.setFormatter(
    logging.Formatter("%(asctime)s %(levelname)s %(message)s")
)
logger.addHandler(file_handler)

logger.info("Started run %s", run_id)
logger.info("Collected %s processed rows", len(df))
logger.info("Processed output path: %s", processed_path)

print(log_path)

## Monitoring Breakage

Monitoring asks whether the run looks operationally plausible. It is not the same as substantive validation.

Useful breakage checks include:

- Did the script collect zero rows?
- Did the row count suddenly drop or explode?
- Are required columns missing?
- Did many requests return `403`, `404`, `429`, or `500` status codes?
- Are important fields unexpectedly empty?
- Did the HTML/API response shape change?

The checks below are deliberately simple, but they teach the pattern.

In [ ]:
required_columns = ["query", "pageid", "title"]
monitoring_checks = []

# Row-count check: empty outputs are one of the most common signs that a scraper
# selector broke or an API request returned something unexpected.
monitoring_checks.append(
    {
        "check": "minimum_row_count",
        "passed": len(df) >= 1,
        "observed": len(df),
        "expected": ">= 1",
    }
)

# Required-column checks: if a column is missing, the parsing/flattening step may
# have changed or failed.
for column in required_columns:
    monitoring_checks.append(
        {
            "check": f"required_column:{column}",
            "passed": column in df.columns,
            "observed": ",".join(df.columns),
            "expected": column,
        }
    )

# Missing-title check: the column may exist, but values may still be empty.
monitoring_checks.append(
    {
        "check": "missing_titles",
        "passed": df["title"].notna().all() if "title" in df.columns else False,
        "observed": int(df["title"].isna().sum()) if "title" in df.columns else "title column missing",
        "expected": "0 missing titles",
    }
)

monitoring_report = {
    "passed": all(check["passed"] for check in monitoring_checks),
    "checks": monitoring_checks,
}

monitoring_path = run_reports_dir / "monitoring_report.json"
pd.Series(monitoring_report).to_json(monitoring_path, indent=2)

monitoring_report

## Versioning

A reproducible run should record not only the data and parameters, but also the code state.

Useful versioning fields include:

- script name;
- command-line arguments or config file;
- Git commit hash;
- Git status, because uncommitted changes matter;
- Python version;
- package versions.

This does not make the run perfectly reproducible, but it makes the conditions of the run inspectable.

In [ ]:
import platform
import subprocess
import sys


def safe_command(command):
    # subprocess.run(...) runs a terminal command from Python.
    # capture_output=True keeps stdout/stderr available as strings.
    # If a command fails, we return None instead of crashing the notebook.
    try:
        result = subprocess.run(
            command,
            check=True,
            capture_output=True,
            text=True,
        )
        return result.stdout.strip()
    except Exception:
        return None


version_info = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "git_commit": safe_command(["git", "rev-parse", "HEAD"]),
    "git_status_short": safe_command(["git", "status", "--short"]),
    # In large projects, pip freeze can be long. For a teaching manifest, it is
    # useful because it records the package environment used for the run.
    "pip_freeze": safe_command([sys.executable, "-m", "pip", "freeze"]),
}

version_path = run_reports_dir / "version_info.json"
pd.Series(version_info).to_json(version_path, indent=2)

print("Git commit:", version_info["git_commit"])
print(version_path)

## Scheduling Collectors

Scheduling means running a collector automatically at a defined time. This can be useful for repeated data collection, but it also increases risk.

Before scheduling a scraper or API collector, ask:

- Is the access route allowed for repeated collection?
- Is the collection frequency necessary for the research question?
- Does the script respect rate limits and server load?
- Does it write logs and monitoring reports?
- Who will notice if it breaks?
- How will credentials/secrets be handled?

Three common scheduling options are cron, launchd, and GitHub Actions.

## Cron Jobs

`cron` is a scheduler on Linux/macOS. You edit scheduled commands with:

```bash
crontab -e
```

Cron syntax is:

```text
minute hour day-of-month month day-of-week command
```

Example: run every Monday at 09:00:

```cron
0 9 * * 1 cd /path/to/repo && python scripts/runnable_workflows/03b_methodsnet_course_scraper.py --details 3 --outdir data >> data/cron.log 2>&1
```

Important teaching points:

- Cron has a minimal environment. `python` may not be the same Python you use in VS Code.
- Use full paths if needed.
- Redirect output to a log file.
- Start with a low-frequency schedule.
- Do not use cron to intensify collection beyond what is justified and allowed.

## macOS launchd

On macOS, `launchd` is often more appropriate than cron. It uses `.plist` files in:

```text
~/Library/LaunchAgents/
```

A LaunchAgent can run a command at a scheduled time and send stdout/stderr to log files.

For teaching, students do not need to memorize the XML. The key point is that scheduling should still call the same reproducible command-line workflow and should still write logs, monitoring reports, and manifests.

In [ ]:
launchd_example = """
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE plist PUBLIC "-//Apple//DTD PLIST 1.0//EN"
 "http://www.apple.com/DTDs/PropertyList-1.0.dtd">
<plist version="1.0">
<dict>
  <key>Label</key>
  <string>org.methodsnet.collection</string>
  <key>ProgramArguments</key>
  <array>
    <string>/bin/zsh</string>
    <string>-lc</string>
    <string>cd /path/to/repo && python scripts/runnable_workflows/03b_methodsnet_course_scraper.py --details 3 --outdir data</string>
  </array>
  <key>StartCalendarInterval</key>
  <dict>
    <key>Weekday</key><integer>1</integer>
    <key>Hour</key><integer>9</integer>
    <key>Minute</key><integer>0</integer>
  </dict>
  <key>StandardOutPath</key>
  <string>/path/to/repo/data/launchd_stdout.log</string>
  <key>StandardErrorPath</key>
  <string>/path/to/repo/data/launchd_stderr.log</string>
</dict>
</plist>
"""

print(launchd_example[:900])

## GitHub Actions

GitHub Actions can run scheduled workflows in a repository. This can be useful for public or institutional projects, but it should be used carefully:

- secrets must be stored as GitHub secrets, not in code;
- scheduled jobs should be small and justified;
- outputs/artifacts need a retention plan;
- monitoring failures should notify someone;
- platform terms/API rules still apply.

In [ ]:
github_actions_example = """
name: Scheduled collection

on:
  schedule:
    - cron: "0 7 * * 1"
  workflow_dispatch:

jobs:
  collect:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -r requirements.txt
      - run: python scripts/runnable_workflows/03b_methodsnet_course_scraper.py --details 3 --outdir data
"""

print(github_actions_example)

## Practical Wrapper Script

The repository now includes a runnable workflow for this operational layer:

```bash
python scripts/runnable_workflows/09_practical_collection_workflow.py   --run-label demo_collection   --outdir /tmp/methodsnet_practical_runs
```

It creates a run folder with:

- `config/config_snapshot.json`
- `raw/`
- `processed/records.csv`
- `logs/run.log`
- `reports/monitoring_report.json`
- `reports/version_info.json`
- `reports/scheduling_examples.md`
- `reports/manifest.json`

This wrapper can also monitor an existing processed CSV, for example a MethodsNET scraper output.

# Exercise

Adapt the project:

1. Change the query.
2. Change the page size.
3. Add one new processed field.
4. Add one limitation to the manifest.
5. Explain what someone would need to reproduce your run.
6. Add one monitoring check that would catch a broken selector or empty API response.
7. Write a cron expression for running a collector once per week.
8. Explain why scheduling without monitoring is risky.

# Key Takeaway

A reproducible collection workflow is not just code that runs. It is a small
project structure that preserves parameters, raw evidence, processed outputs,
logs, monitoring checks, version information, and limitations.

Scheduling is the last step, not the first. A collector should only be scheduled
after it is documented, rate-limited, monitored, and compliant with the relevant
access rules.